## CSV Data Ingestion & Parser

In [ ]:
import pandas as pd
import time

In [ ]:
data = pd.read_csv('../data/raw/portfolio - portfolio.csv')

In [ ]:
data.head()

In [ ]:
data.info()

### Existence Checks

In [ ]:
MUST_HAVE_COLUMNS = ["Portfolio", "Transaction Date", "Transaction Type", "Ticker", "Quantity", "Transaction Price", "Fee"]

missing_columns = []
for col in MUST_HAVE_COLUMNS:
    if col not in data.columns:
        missing_columns.append(col)

if len(missing_columns) > 0:
    raise ValueError(f"The following columns are missing from the data: {missing_columns}")

data = data[MUST_HAVE_COLUMNS]

In [ ]:
data.info()

### Data Type Checks

In [ ]:
MUST_HAVE_DATA_TYPES = {
    "Portfolio": lambda x: isinstance(x, str) or isinstance(x, int),                        # string or integer
    "Transaction Date": lambda x: 
        (pd.to_datetime(x, format='%Y%m%d', errors='coerce') is not pd.NaT) or 
        (pd.to_datetime(x, format='%Y-%m-%d', errors='coerce') is not pd.NaT),              # format should match YYYY-MM-DD or YYYYMMDD
    "Transaction Type": lambda x: isinstance(x, str),                                       # string
    "Ticker": lambda x: isinstance(x, str),                                                 # string
    "Quantity": lambda x: isinstance(x, int),                                               # integet
    "Transaction Price": lambda x: (isinstance(x, float) or isinstance(x, int)),            # float or integer
    "Fee": lambda x: (isinstance(x, float) or isinstance(x, int))                           # float or integer
}

data_types_violated = []
for col in MUST_HAVE_DATA_TYPES:
    if not all(data[col].apply(MUST_HAVE_DATA_TYPES[col])):
        data_types_violated.append(col)

if data_types_violated:
    raise ValueError(f"Some values in columns {data_types_violated} do not meet the required data types.")

# convert data types to the required 
MUST_HAVE_DATA_TYPES = {
    "Portfolio": str, 
    "Transaction Date": str,
    "Transaction Type": str,
    "Ticker": str,
    "Quantity": int,
    "Transaction Price": float,
    "Fee": float
}
data = data.astype(MUST_HAVE_DATA_TYPES)

# specific handling of date column
if any(data["Transaction Date"].apply(lambda x: (pd.to_datetime(x, format='%Y%m%d', errors='coerce') is not pd.NaT))):
    data["Transaction Date"] = pd.to_datetime(data["Transaction Date"], format='%Y%m%d', errors='coerce')
else:
    data["Transaction Date"] = pd.to_datetime(data["Transaction Date"], format='%Y-%m-%d', errors='coerce')

# specific handling of asset column
data["Ticker"] = data["Ticker"].str.strip().str.upper()

In [ ]:
data.info()

### Content Validity Checks

In [ ]:
MUST_HAVE_DATA_RULES = {
    "Portfolio": lambda x: len(x) > 0,                                                  # at least one character long
    "Transaction Date": lambda x: x <= pd.to_datetime(time.ctime()),                    # date before current time
    "Transaction Type": lambda x: x in ["BUY", "SELL"],                                 # available values: BUY, SELL
    "Ticker": lambda x: len(x) > 0,                                                     # at least one character long
    "Quantity": lambda x: x > 0,                                                        # positive number
    "Transaction Price": lambda x: x >= 0,                                              # non-negative number
    "Fee": lambda x: x >= 0                                                             # non-negative number
}

rules_violated = []
for col in MUST_HAVE_DATA_RULES:
    if not all(data[col].apply(MUST_HAVE_DATA_RULES[col])):
        rules_violated.append(col)

if rules_violated:
    raise ValueError(f"Some values in columns {rules_violated} do not meet the required rules.")

In [ ]:
data.head()

## Market Data Integration

In [42]:
import pandas as pd
import yfinance as yf
import time

from src.backend.file_handler import load_transactional_data, check_required_columns, check_data_types, check_data_validity

In [24]:
filename = "portfolio - portfolio.csv"

data = load_transactional_data(filename)
data = check_required_columns(data)
data = check_data_types(data)
data = check_data_validity(data)

data.head()

✅ File loaded successfully!

✅ File contains all required columns!

✅ Passed all data type rules!

✅ Passed all data validity rules!



,Portfolio,Transaction Date,Transaction Type,Ticker,Quantity,Transaction Price,Fee
0,1,2026-08-10,BUY,VUAA.L,4,150.06,2.07
1,1,2026-07-10,BUY,VUAA.L,4,145.72,2.07
2,1,2026-06-11,BUY,VUAA.L,4,141.40,2.07
3,1,2026-05-12,BUY,VUAA.L,2,142.30,2.03
4,1,2026-04-20,BUY,VUAA.L,4,136.52,2.07


In [25]:
tickers_unique = data["Ticker"].unique().tolist()

In [26]:
tickers_unique

['VUAA.L',
 'VUSA.L',
 'VHYD.L',
 'DGRW.L',
 'AETF.AT',
 'GRE.PA',
 'XEON.DE',
 'SVXY']

In [54]:
tickers = yf.Tickers(tickers_unique)

In [81]:
tickers.tickers

{'VUAA.L': yfinance.Ticker object <VUAA.L>,
 'VUSA.L': yfinance.Ticker object <VUSA.L>,
 'VHYD.L': yfinance.Ticker object <VHYD.L>,
 'DGRW.L': yfinance.Ticker object <DGRW.L>,
 'AETF.AT': yfinance.Ticker object <AETF.AT>,
 'GRE.PA': yfinance.Ticker object <GRE.PA>,
 'XEON.DE': yfinance.Ticker object <XEON.DE>,
 'SVXY': yfinance.Ticker object <SVXY>}

In [82]:
value_dict = tickers.history(period="1d")["Close"].to_dict()
value_dict = {key: next(iter(values.values())) for key, values in value_dict.items()}

[*********************100%***********************]  8 of 8 completed


In [83]:
value_dict

{'AETF.AT': 67.37999725341797,
 'DGRW.L': 51.709999084472656,
 'GRE.PA': 2.933500051498413,
 'SVXY': 60.790000915527344,
 'VHYD.L': 94.69999694824219,
 'VUAA.L': 150.39999389648438,
 'VUSA.L': 108.91500091552734,
 'XEON.DE': 149.9219970703125}